# Deep ResearchAgents

**Module 11 · Lesson 11.8**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Deep-Research Loop: What It Actually IS
- Decomposition & Planning: Parallel or Sequential?
- Search & Source Evaluation: Not All Sources Are Equal
- Grounding & Citations: The Hardest Problem
- Synthesis: Map-Reduce and the Faithfulness Tax
- The Production Landscape: Four Bets on the Same Loop

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install "anthropic>=0.40.0" python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com


## “Should we adopt Framework X, and what will a year cost us?”

> 💡 **Info**
>
> A search engine hands you ten blue links and leaves the reading, the cross-checking, and the writing to you. A **deep-research agent** does the whole job: it splits the question into sub-questions (what does X do? what does it cost? who else uses it? what breaks?), searches a dozen sources, judges which to trust, pulls the key facts, notices where two sources disagree, and writes a **cited, structured report** - in minutes. That is genuinely useful. But there is a trap: if the agent cites a number its source never actually said - and a *large fraction* of LLM citations do exactly that - then the confident, well-formatted report is *worse* than no report, because you will act on it. This lesson builds the loop AND the grounding that keeps it honest.

### 🎯 What you will be able to do after this lesson

- **Build** the deep-research loop - decompose, search, evaluate, extract, cross-check, synthesize - end to end.

- **Compare** decomposition strategies (parallel vs sequential) and the four production architectures (OpenAI / Google / Anthropic / Perplexity).

- **Operate** the grounding spine: source-credibility scoring and citation verification that catches unsupported claims.

- **Evaluate** the synthesis faithfulness tradeoff and the cost and stopping controls that make deep research shippable.

> 📦 **Info**
>
> ✅ Before you startThis builds on **11.1** (the agent loop: observe → think → act), **11.5** (the multi-agent tax - deep research is where it pays off), and **9.2** (Reflexion - the gap-analysis step is self-correction applied to research). This lesson is the deep-research **architecture**; deep evaluation of research agents is Lesson 11.11 / Module 14, and Indian-language search is Lesson 17.3.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 📚 **Analogy**
>
> A search engine is a **librarian pointing at a shelf**: here are the books, good luck. A deep-research agent is a **PhD student** who breaks your question into five sub-questions, reads fifteen papers, takes notes on each, checks which sources agree and which contradict, and writes you a structured, cited summary. **Where it breaks down:** a real PhD student would never invent a citation; an agent will confidently cite a source that never said the thing - which is why grounding (Step 4) is the heart of the lesson, not a footnote.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Deep research is just search with more steps.”** A search engine returns links; a deep-research agent returns a *cited answer* built from decomposition, cross-checking, and synthesis. The quality is in the loop, not a clever prompt.
> - **“Trust the model’s citations.”** A large fraction of LLM citations do not support their claim. Without a verification layer, a confident cited report can be mostly fabricated.

> 💡 **Info**
>
> ⚠️ Anti-patternShipping a cited-looking report with no citation-verification layer, and running the iterative loop with no stopping criteria. The first makes the agent authoritative and wrong; the second makes it search forever while the bill climbs. Ground every claim, and cap coverage, novelty, iterations, and budget.

---

## 🎯 Concept 1: The Deep-Research Loop: What It Actually IS

### The Deep-Research Loop: What It Actually IS

Decompose -> search -> extract -> cross-check -> synthesize. A search engine returns links; this returns a cited answer.

Start with the whole loop, because every later step is one piece of it. A deep-research agent takes one hard question and runs a pipeline: **decompose** it into focused sub-questions, **search** sources for each, **extract** the key findings, **cross-check** for agreements and contradictions, and **synthesize** a cited report. The crucial mental model (carried from 11.1): the loop learns from *ground truth* - what the sources actually say - not from the model’s own confidence. The block below runs the entire shape with a scripted “LLM” and a tiny mock “web,” so you can watch a question become a cited answer with no API key.

> 🧠 **Analogy**
>
> It is the difference between a **vending machine** and a **chef**. A search engine (vending machine) hands you exactly what you punched in. A deep-research agent (chef) takes your request, decides what ingredients it needs, gathers them from several places, tastes as it goes, and plates a finished dish. You asked a question; it hands back an answer, not a pile of ingredients.

What is the single biggest difference between a deep-research agent and a Google search?

**📝 `01_research_loop.py`** - *runnable*

In [ ]:
# THE DEEP-RESEARCH LOOP - what a research agent IS (vs a search engine).
# A search engine returns LINKS. A deep-research agent returns a CITED ANSWER by
# looping: decompose -> search -> extract -> cross-check -> synthesize. We script
# the "LLM" and the "web" so the whole SHAPE runs with no API key.

def fake_llm_decompose(question):             # THINK: split into focused sub-questions
    return ["what does it cover", "what does it cost", "what career outcomes", "what do reviews say"]

WEB = {                                        # a tiny mock corpus (source -> text)
    "cover":    [("netsetos.com", "18 modules, 150 hours, hands-on projects.")],
    "cost":     [("netsetos.com", "14999 INR."), ("coursera.org", "similar track ~24500 INR.")],
    "career":   [("linkedin.com", "GenAI roles 15-40 LPA in India.")],
    "reviews":  [("reddit.com", "practical, project-heavy."), ("reddit.com", "self-paced may be enough.")],
}

def search(sub_q):                             # ACT: return sources for a sub-question
    for key, hits in WEB.items():
        if key in sub_q or key[:4] in sub_q:
            return hits
    return []

def extract(sources):                          # pull findings + note contradictions
    findings = [f"{s}: {t}" for s, t in sources]
    contradictions = [t for s, t in sources if "self-paced" in t]   # a source that pushes back
    return findings, contradictions

def synthesize(question, findings, contradictions):
    body = " ".join(f"[{f.split(':')[0]}]" for f in findings)       # a cited report stub
    note = " Contradiction noted." if contradictions else ""
    return f"Report on '{question}': grounded in {body}.{note}"

def research(question):
    subs = fake_llm_decompose(question)
    all_sources, all_findings, all_contra = [], [], []
    for sq in subs:
        src = search(sq)
        all_sources += src
        f, c = extract(src)
        all_findings += f; all_contra += c
    report = synthesize(question, all_findings, all_contra)
    return subs, all_sources, all_findings, all_contra, report

q = "Is the GenAI course worth it for a working professional?"
subs, sources, findings, contra, report = research(q)
print("Deep-research loop (decompose -> search -> extract -> synthesize):")
print(f"  sub-questions : {len(subs)}")
print(f"  sources found : {len(sources)}")
print(f"  findings      : {len(findings)}  | contradictions: {len(contra)}")
print(f"  report        : {report}")
print("A search engine would hand you the links; the agent hands you the answer.")
# Output:
# Deep-research loop (decompose -> search -> extract -> synthesize):
#   sub-questions : 4
#   sources found : 6
#   findings      : 6  | contradictions: 1
#   report        : Report on 'Is the GenAI course worth it for a working professional?': grounded in [netsetos.com] [netsetos.com] [coursera.org] [linkedin.com] [reddit.com] [reddit.com]. Contradiction noted.
# A search engine would hand you the links; the agent hands you the answer.

- `fake_llm_decompose` splits the question into focused sub-questions (the THINK step).
- `search` returns sources per sub-question (the ACT step); `extract` pulls findings AND flags a contradiction.
- `synthesize` writes a cited report stub from the findings - an answer, not a link list.
- The whole pipeline runs with no key because the “LLM” and “web” are scripted; the SHAPE is what matters.

#### 💡 What just happened

⚡ What just happened? A deep-research agent is a loop: decompose -> search -> extract -> cross-check -> synthesize, learning from what the sources say. Tradeoff: far more useful than links, but slower and more expensive, and only as trustworthy as its grounding. The next steps make each stage real.

- Step a question through decompose -> search -> evaluate -> extract -> cross-check -> synthesize
- A search engine returns links; this returns a cited answer

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Decomposition & Planning: Parallel or Sequential?

### Decomposition & Planning: Parallel or Sequential?

Split the question into sub-questions - then route the independent ones in parallel and the dependent ones in order.

The first stage, decomposition, hides a planning decision. Some sub-questions are **independent** (“what does it cost” and “what does it cover” can be searched at the same time) and should run in **parallel** for speed. Others are **dependent** (“find the CEO” then “research *that* person’s background”) and *must* run in **order**, because the second query needs the first answer. Parallelize a dependent pair and you waste tokens searching for something you cannot search yet. The guidance (from LangChain and NVIDIA’s ParallelSearch work): bias toward a single sub-agent, and only fan out when the query is genuinely a comparison or has clearly independent parts. The block plans a set of sub-questions by dependency.

> 🤳 **Analogy**
>
> It is **assigning a group project**. “Research the market” and “research the competitors” can be handed to two people at once (parallel). But “interview the winner” cannot start until “pick the winner” is done (sequential). A good project lead splits the independent work and chains only what truly must wait.

Sub-question B needs sub-question A’s answer as its input. Can they run in parallel?

**📝 `02_plan_parallel.py`** - *runnable*

In [ ]:
# DECOMPOSE + PLAN - not every sub-question can run in parallel.
# INDEPENDENT sub-questions (compare X vs Y) run in PARALLEL; DEPENDENT ones
# (find the CEO, THEN research their background) must run in ORDER. Route by dependency.

SUBQUESTIONS = [
    {"id": 1, "q": "what does the course cover",        "depends_on": None},
    {"id": 2, "q": "what does it cost vs alternatives", "depends_on": None},
    {"id": 3, "q": "what career outcomes do grads get", "depends_on": None},
    {"id": 4, "q": "for THOSE outcomes, the salary range", "depends_on": 3},   # needs #3 first
]

def plan(subs):
    parallel = [s for s in subs if s["depends_on"] is None]
    dependent = [s for s in subs if s["depends_on"] is not None]
    return parallel, dependent

def run(subs):
    parallel, dependent = plan(subs)
    order = []
    order.append(("PARALLEL", [s["id"] for s in parallel]))   # fan out independent sub-qs at once
    for s in dependent:                                       # then the ones that had to wait
        order.append(("AFTER #%d" % s["depends_on"], [s["id"]]))
    return order, len(parallel), len(dependent)

order, n_par, n_dep = run(SUBQUESTIONS)
print("Plan the sub-questions by dependency:")
for stage, ids in order:
    print(f"  {stage:10} -> sub-questions {ids}")
print(f"\n{n_par} run in parallel, {n_dep} waits on its dependency.")
naive = len(SUBQUESTIONS)              # sequential-only would run all 4 one after another
smart = 1 + n_dep                      # 1 parallel wave + the dependent tail
print(f"Sequential-only: {naive} waves. Dependency-aware: {smart} waves (parallel where safe).")
# Output:
# Plan the sub-questions by dependency:
#   PARALLEL   -> sub-questions [1, 2, 3]
#   AFTER #3   -> sub-questions [4]
#
# 3 run in parallel, 1 waits on its dependency.
# Sequential-only: 4 waves. Dependency-aware: 2 waves (parallel where safe).

- `plan` separates sub-questions with no dependency from the one that has a `depends_on`.
- The independent sub-questions are dispatched in one **parallel** wave; the dependent one waits for its parent.
- Dependency-aware planning finishes in 2 waves where sequential-only would take 4 - parallel where it is safe.
- Rule of thumb: default to one sub-agent, and only parallelize genuinely independent work.

#### 💡 What just happened

⚡ What just happened? Decomposition is also a plan: independent sub-questions run in parallel, dependent ones in order. Tradeoff: parallelism is faster but only correct for independent sub-questions; premature parallelization wastes tokens on searches that must be redone. Plan by dependency, not by reflex.

- A question splits into sub-questions; independent ones fan out in parallel
- A dependent sub-question waits on its parent; watch the wrong-order cost

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Search & Source Evaluation: Not All Sources Are Equal

### Search & Source Evaluation: Not All Sources Are Equal

Choose a search API for the query, then score each source by credibility tier and recency - and drop the weak ones.

Now the search stage. First, *which* search? The API landscape shifted in 2025-26: **Microsoft retired the Bing Search API in August 2025**, so agents moved to purpose-built APIs - **AI-native** ones (Tavily, Exa, Firecrawl) that return LLM-ready content, **SERP scrapers** (Serper) that return cheap Google snippets, and **readers** (Jina) that turn a URL into clean Markdown. Then, once you have results, comes the part that separates a good agent from a plausible-sounding one: **source evaluation**. A peer-reviewed study and a random five-year-old blog are not equal evidence. Score each source by a **credibility tier** (.gov and journals over news over blogs over social) times a **recency** factor, and drop what falls below the bar. The runnable block scores a mixed source set; the illustrative block shows the current search APIs.

> ⚖️ **Analogy**
>
> It is a **judge weighing witnesses**. A forensic report from a lab (a .gov study) carries more weight than an anonymous tip (a social post), and a fresh account beats a decade-old memory. The agent is not just collecting testimony - it is deciding whose testimony to believe, and how much.

A fresh anonymous social-media post and a two-year-old peer-reviewed journal article. Which should score higher?

**📝 `03_source_eval.py`** - *runnable*

In [ ]:
# SOURCE EVALUATION - not all sources are equal. Score each by a CREDIBILITY
# TIER (a .gov study outranks a random blog) and RECENCY (fresh beats stale),
# then keep only what clears the bar. This is what stops a confident wrong report.

TODAY = 2026
TIER = {"gov": 5, "journal": 5, "news": 3, "industry": 3, "blog": 2, "social": 1}

SOURCES = [
    {"url": "nist.gov/report",     "tier": "gov",      "year": 2026, "text": "official benchmark data"},
    {"url": "random-blog.com",     "tier": "blog",     "year": 2021, "text": "one person's old opinion"},
    {"url": "reuters.com/story",   "tier": "news",     "year": 2026, "text": "reported figures"},
    {"url": "x.com/anon",          "tier": "social",   "year": 2026, "text": "unverified claim"},
    {"url": "acm.org/paper",       "tier": "journal",  "year": 2024, "text": "peer-reviewed study"},
]

def recency(year):                              # linear decay, floored at 0.2; newer scores higher
    age = TODAY - year
    return max(0.2, 1.0 - 0.15 * age)           # newer -> closer to 1.0

def score(s):
    return round(TIER[s["tier"]] * recency(s["year"]), 2)

ranked = sorted(SOURCES, key=score, reverse=True)
KEEP = 3.0                                       # drop anything below the bar
print("Score sources by credibility tier x recency:")
for s in ranked:
    verdict = "KEEP " if score(s) >= KEEP else "DROP "
    print(f"  {verdict} {score(s):>4}  {s['tier']:<8} {s['year']}  {s['url']}")
kept = [s for s in ranked if score(s) >= KEEP]
print(f"\nKept {len(kept)} of {len(SOURCES)} sources; a fresh .gov beats a stale blog and a social post.")
# Output:
# Score sources by credibility tier x recency:
#   KEEP   5.0  gov      2026  nist.gov/report
#   KEEP   3.5  journal  2024  acm.org/paper
#   KEEP   3.0  news     2026  reuters.com/story
#   DROP   1.0  social   2026  x.com/anon
#   DROP   0.5  blog     2021  random-blog.com
#
# Kept 3 of 5 sources; a fresh .gov beats a stale blog and a social post.

- Each source gets a `TIER` weight (.gov / journal highest, social lowest) times a `recency` factor.
- Sorting by that score ranks a fresh .gov report above a stale blog and an anonymous social post.
- Anything below the `KEEP` bar is dropped, so weak evidence never reaches synthesis.
- In production the tiers are a policy you tune; the LLM can add a content-quality score on top.

**📝 `03b_search_apis.py`** - *illustrative*

In [ ]:
# SEARCH APIS FOR AGENTS - the 2026 landscape (illustrative minimal example).
# Microsoft retired the Bing Search API in Aug 2025, so agents moved to purpose-built
# APIs. Three kinds: AI-native (extract content), SERP scrapers (snippets only),
# and readers (URL -> clean Markdown). Mix them by query type and budget.
import os, requests   # pip install requests

def tavily_search(query, k=5):        # AI-native: structured, LLM-ready, includes content
    r = requests.post("https://api.tavily.com/search", json={
        "api_key": os.environ["TAVILY_API_KEY"], "query": query,
        "search_depth": "advanced", "max_results": k, "include_raw_content": True})
    return r.json()["results"]        # [{title, url, content, score}, ...]

def serper_then_jina(query, k=5):     # BUDGET: cheap Google SERP + free Markdown reader
    serp = requests.post("https://google.serper.dev/search",
        headers={"X-API-KEY": os.environ["SERPER_API_KEY"]},
        json={"q": query, "num": k}).json()
    pages = []
    for hit in serp.get("organic", [])[:k]:
        md = requests.get("https://r.jina.ai/" + hit["link"]).text   # prepend r.jina.ai/ -> Markdown
        pages.append({"url": hit["link"], "markdown": md})
    return pages

# Route by need: Tavily/Exa for agent-ready search, Serper+Jina for cheap high-volume,
# Firecrawl for JS-heavy pages, Brave for an independent index. There is no single winner.
# Output: (illustrative minimal example - needs TAVILY_API_KEY / SERPER_API_KEY + `pip install requests`.)
# tavily_search returns [{title, url, content, score}, ...]; serper_then_jina returns
# [{url, markdown}, ...] - clean text ready for the extract + synthesize steps.

- `tavily_search` is AI-native: structured, LLM-ready results with content included.
- `serper_then_jina` is the budget pipeline: cheap Google SERP + `r.jina.ai/` for free Markdown extraction.
- Bing’s API retired in Aug 2025, so agents mix these purpose-built APIs by query type and budget.
- There is no single winner - Tavily/Exa for agent-ready search, Serper+Jina for volume, Firecrawl for JS pages, Brave for an independent index.

#### 💡 What just happened

⚡ What just happened? Search has two decisions: which API (AI-native vs SERP vs reader, since Bing’s API is gone), and which sources to trust (credibility tier times recency). Tradeoff: broad search finds more, but unfiltered sources let a stale blog outvote a study. Score and filter before you synthesize.

- Score sources on credibility tier x recency
- Low-credibility and stale sources drop out of the working set

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Grounding & Citations: The Hardest Problem

### Grounding & Citations: The Hardest Problem

A large fraction of LLM citations do not support their claim. The fix is a verification layer, not a better prompt.

This is the step that decides whether the whole report is trustworthy. Studies in 2026 find LLM citation-hallucination rates ranging from roughly **15% to over 50%** (and far higher in specialized domains like law) - the model attaches a citation to a claim the source never actually made. You cannot prompt your way out of it. The production fix is a **three-layer defense**: (1) store each source as a chunk with metadata (url, title, chunk id); (2) have the generator reference chunk ids as it writes; (3) run a separate **verification agent** (Anthropic’s CitationAgent pattern) that checks each claim against its cited chunk and flags the ones that do not hold. Stanford-style multi-layer validation drives hallucination to **under one percent**, versus roughly 17 to 33 percent for retrieval alone. The block runs the verifier over a drafted report.

> 🔍 **Analogy**
>
> It is a **fact-checker at a newspaper**. The reporter writes “a study found X (see source 3).” The fact-checker opens source 3 and reads it: does it actually say X? If yes, it ships; if it says something weaker, it gets hedged; if source 3 never mentions X, the line is cut. No claim reaches print on the reporter’s word alone.

A cited report looks confident and well-formatted. Can you trust its citations without checking?

**📝 `04_citation_verify.py`** - *runnable*

In [ ]:
# GROUNDING + CITATION VERIFICATION - the hardest problem in deep research.
# A large fraction of LLM citations do not actually support their claim. The fix is
# not a better prompt: store source chunks, then VERIFY each claim against its chunk
# and refuse to ship the ones the source never said.

CHUNKS = {                                       # chunk_id -> the exact source text
    "c1": "The course has 18 modules and 150 hours of content.",
    "c2": "Tuition is 14999 INR with EMI available.",
    "c3": "Graduates report roles paying 15 to 40 LPA in India.",
}

# What the synthesis LLM drafted: each claim points at a chunk it says supports it.
DRAFT = [
    {"claim": "It has 18 modules.",              "cites": "c1"},
    {"claim": "Tuition is 14999 INR.",           "cites": "c2"},
    {"claim": "It guarantees a 40 LPA job.",     "cites": "c3"},   # source says "report", not "guarantee"
    {"claim": "It includes 5000 GCP credits.",   "cites": "c2"},   # chunk never mentions credits
    {"claim": "It offers about 200 hours of content.", "cites": "c1"},  # chunk says 150, not 200
]

def verify(claim, chunk_text):
    # A real system uses an NLI / LLM entailment check; here a keyword-overlap stand-in.
    import re
    key = [w for w in re.findall(r"[a-z0-9]+", claim.lower()) if len(w) > 3 and w not in ("with","that","this")]
    hits = sum(1 for w in key if w in chunk_text.lower())
    if hits == len(key) and key:            return "SUPPORTED"
    if "guarantee" in claim.lower():         return "UNSUPPORTED"   # source hedges, claim over-states
    if hits >= 1:                            return "PARTIAL"
    return "UNSUPPORTED"

print("Verify every cited claim against its source chunk:")
counts = {"SUPPORTED": 0, "PARTIAL": 0, "UNSUPPORTED": 0}
for d in DRAFT:
    verdict = verify(d["claim"], CHUNKS[d["cites"]])
    counts[verdict] += 1
    print(f"  [{verdict:<11}] {d['claim']}  (cites {d['cites']})")
print(f"\nShip the {counts['SUPPORTED']} SUPPORTED; hedge the {counts['PARTIAL']} PARTIAL; drop the {counts['UNSUPPORTED']} UNSUPPORTED.")
print("Without this layer a confident, cited report can be mostly fabricated.")
# Output:
# Verify every cited claim against its source chunk:
#   [SUPPORTED  ] It has 18 modules.  (cites c1)
#   [SUPPORTED  ] Tuition is 14999 INR.  (cites c2)
#   [UNSUPPORTED] It guarantees a 40 LPA job.  (cites c3)
#   [UNSUPPORTED] It includes 5000 GCP credits.  (cites c2)
#   [PARTIAL    ] It offers about 200 hours of content.  (cites c1)
#
# Ship the 2 SUPPORTED; hedge the 1 PARTIAL; drop the 2 UNSUPPORTED.
# Without this layer a confident, cited report can be mostly fabricated.

- Each drafted claim points at a source `chunk`; `verify` checks the claim against that chunk’s text.
- A claim the chunk fully supports is `SUPPORTED`; a hedge overstated as a “guarantee” and an invented credit are `UNSUPPORTED`.
- “200 hours” against a “150 hours” chunk is `PARTIAL` - flagged to hedge, not shipped as fact.
- Real systems use an NLI/entailment model here; the keyword check is a keyless stand-in for the SAME architecture.

#### 💡 What just happened

⚡ What just happened? Grounding is architecture, not a prompt: store chunks, reference chunk ids, and verify every claim against its source (the three-layer defense). Tradeoff: verification costs an extra pass, but without it a confident cited report can be mostly fabricated. This is the heart of a trustworthy research agent.

- Trace each claim back to its source chunk
- Unsupported claims flash red; the three-layer defense drives the fabrication rate down

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Synthesis: Map-Reduce and the Faithfulness Tax

### Synthesis: Map-Reduce and the Faithfulness Tax

Fold many source summaries into one report - without drifting from the facts. Quote the critical numbers verbatim.

With grounded findings in hand, the agent writes the report. The standard pattern is **map-reduce**: in the *map* step, summarize each source (parallelizable); in the *reduce* step, synthesize the summaries into one report. The catch is a **faithfulness tax**: every abstraction step is a chance to drift from the source, and research finds a *negative* correlation between how abstractive a summary is and how faithful it stays. A smoothly-written, highly-paraphrased report is exactly the one most likely to have quietly changed a number. The fix is **extract-then-abstract**: quote the critical facts verbatim with a citation, and paraphrase only the connective narrative. The block runs map-reduce at two abstractiveness levels and shows the faithful one keep its numbers and the over-abstracted one lose them.

> 📝 **Analogy**
>
> It is writing a **literature review**. You quote the exact figures a study reported - verbatim, with the citation - and paraphrase only the story that connects them. Rewrite the numbers “in your own words” and you will eventually turn a precise result into “most,” then “the vast majority,” and the fact is gone. Quote what matters; paraphrase the glue.

Which report is MORE likely to have quietly changed a fact?

**📝 `05_map_reduce.py`** - *runnable*

In [ ]:
# SYNTHESIS: MAP-REDUCE + FAITHFULNESS - combine many sources into one report
# without drifting from the facts. MAP: summarize each source. REDUCE: synthesize.
# The trap: the more you abstract, the LESS faithful; so QUOTE critical facts verbatim.

SOURCES = [
    {"src": "netsetos.com", "text": "18 modules, 150 hours.", "fact": "150 hours"},
    {"src": "coursera.org", "text": "comparable track around 24500 INR.", "fact": "24500 INR"},
    {"src": "linkedin.com", "text": "GenAI roles pay 15 to 40 LPA in India.", "fact": "15-40 LPA"},
    {"src": "reddit.com",   "text": "learners call it practical and project-heavy.", "fact": None},
]

def map_summarize(s):                            # MAP: one short summary per source
    return {"src": s["src"], "summary": s["text"][:40], "fact": s["fact"]}

def reduce_report(summaries, abstractiveness):
    # abstractiveness 0..3: higher = more paraphrase = less faithful.
    verbatim = [m["fact"] for m in summaries if m["fact"]]        # critical facts kept EXACT
    report = "Synthesis: the course runs 150 hours; peers cost ~24500 INR; roles pay 15-40 LPA."
    faithful = "150 hours" in report and "15-40 LPA" in report    # the facts survived
    if abstractiveness >= 3:                                       # over-abstraction drops a number
        report = "Synthesis: a fairly long, mid-priced course with solid job prospects."
        faithful = "150 hours" in report
    return report, verbatim, faithful

summaries = [map_summarize(s) for s in SOURCES]
print("MAP - per-source summaries:")
for m in summaries:
    print(f"  [{m['src']}] {m['summary']}  fact={m['fact']}")

for a in (1, 3):
    report, verbatim, faithful = reduce_report(summaries, abstractiveness=a)
    print(f"\nREDUCE (abstractiveness={a}): faithful={faithful}")
    print(f"  {report}")
print("\nMore abstraction reads smoother but drops facts; extract-then-abstract keeps numbers verbatim.")
# Output:
# MAP - per-source summaries:
#   [netsetos.com] 18 modules, 150 hours.  fact=150 hours
#   [coursera.org] comparable track around 24500 INR.  fact=24500 INR
#   [linkedin.com] GenAI roles pay 15 to 40 LPA in India.  fact=15-40 LPA
#   [reddit.com] learners call it practical and project-h  fact=None
#
# REDUCE (abstractiveness=1): faithful=True
#   Synthesis: the course runs 150 hours; peers cost ~24500 INR; roles pay 15-40 LPA.
#
# REDUCE (abstractiveness=3): faithful=False
#   Synthesis: a fairly long, mid-priced course with solid job prospects.
#
# More abstraction reads smoother but drops facts; extract-then-abstract keeps numbers verbatim.

- `map_summarize` produces one short summary per source, keeping the critical `fact` attached.
- `reduce_report` at low abstractiveness keeps the numbers verbatim - the faithfulness check passes.
- At high abstractiveness the report reads smoother but drops a number - the faithfulness check fails.
- Extract-then-abstract: quote the facts that matter, paraphrase only the connective tissue.

Now the **payoff**: assemble the faithful, verified findings into the cited, structured report the cold-open promised. Reuse only the **SUPPORTED** claims from Step 4, give each source a stable number, and emit an executive summary, findings with inline `[n]` markers, and a reference list - the actual artifact a deep-research agent hands back.

**📝 `05b_assemble_report.py`** - *runnable*

In [ ]:
# ASSEMBLE THE CITED REPORT - the payoff the whole loop exists for. Turn the VERIFIED
# claims (Step 4) into the structured artifact promised in the cold-open: an executive
# summary, findings with inline [n] citations, and a numbered reference list. Ship only
# what was SUPPORTED - the unsupported and partial claims never make it into the report.

SUPPORTED = [                                    # only the claims that passed verification (Step 4)
    {"text": "The course has 18 modules and 150 hours of content.", "src": "netsetos.com"},
    {"text": "Tuition is 14999 INR with EMI available.",            "src": "netsetos.com"},
    {"text": "Comparable peer tracks cost about 24500 INR.",        "src": "coursera.org"},
]

def assemble(question, claims):
    refs = []                                    # numbered, de-duplicated reference list
    def cite(src):
        if src not in refs:
            refs.append(src)
        return refs.index(src) + 1               # the [n] marker for this source
    findings = [f"{c['text']} [{cite(c['src'])}]" for c in claims]
    return {
        "summary": "A 150-hour, 14999 INR program - below comparable peers - so worth it if you will use it.",
        "findings": findings,
        "references": [f"[{i + 1}] {src}" for i, src in enumerate(refs)],
    }

report = assemble("Is the course worth it?", SUPPORTED)
print("EXECUTIVE SUMMARY:")
print("  " + report["summary"])
print("\nFINDINGS (each claim carries an inline citation):")
for line in report["findings"]:
    print("  - " + line)
print("\nREFERENCES:")
for ref in report["references"]:
    print("  " + ref)
print("\nEvery sentence traces to a numbered source - a cited, STRUCTURED report, not a link list.")
# Output:
# EXECUTIVE SUMMARY:
#   A 150-hour, 14999 INR program - below comparable peers - so worth it if you will use it.
#
# FINDINGS (each claim carries an inline citation):
#   - The course has 18 modules and 150 hours of content. [1]
#   - Tuition is 14999 INR with EMI available. [1]
#   - Comparable peer tracks cost about 24500 INR. [2]
#
# REFERENCES:
#   [1] netsetos.com
#   [2] coursera.org
#
# Every sentence traces to a numbered source - a cited, STRUCTURED report, not a link list.

- The report ships **only** the claims that passed verification in Step 4 - nothing unsupported or partial.
- `cite()` assigns each source a stable number and reuses it, building the reference list automatically.
- The result is *structured*: an executive summary, findings each carrying an inline `[n]` citation, and a numbered reference list.
- Every sentence traces back to a source - a cited report, not a pile of links (the cold-open, paid off).

#### 💡 What just happened

⚡ What just happened? Synthesis is map-reduce with a faithfulness tax: more abstraction means less faithfulness, so quote critical facts verbatim and paraphrase only the glue. Tradeoff: a fully-extractive report is dull, a fully-abstractive one drifts; extract-then-abstract is the honest middle. Then verify the synthesized claims (Step 4) once more.

- Per-source summaries (map) fold into one report (reduce)
- An abstractiveness slider trades readability against faithfulness

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: The Production Landscape: Four Bets on the Same Loop

### The Production Landscape: Four Bets on the Same Loop

OpenAI, Google, Anthropic, and Perplexity build the same loop with different strategies - and open-source matches them.

Every commercial deep-research product runs the loop you just built; they differ in the **bet**. **OpenAI Deep Research** is a *single agent* (an o3-class model RL-trained to browse), 30-60 searches over 5-30 minutes - it bets on reasoning depth (about 26.6% on Humanity’s Last Exam at launch, the first tool-using result well above base models). **Google Gemini** shows you a *research plan to approve or edit* before it runs, with deep Workspace integration - it bets on user control. **Anthropic** uses a *multi-agent* orchestrator-worker (an Opus lead delegating to parallel Sonnet workers, plus a CitationAgent) - it beat a single agent by **~90%** at **~15x the tokens**, and found that token usage alone explains ~80% of the performance variance. **Perplexity** is the *fastest* (under 3 minutes) with a pay-as-you-go API - it bets on speed. And you do not need any of them: open-source (LangChain Open Deep Research, GPT Researcher) reaches comparable quality at a fraction of the cost.

> 📰 **Analogy**
>
> It is **four newsrooms covering the same story**. One sends a single star investigator who reads everything deeply (OpenAI). One shows the editor the plan before reporting starts (Google). One runs a bureau of parallel reporters with a fact-checker (Anthropic). One races to publish first (Perplexity). Same craft, four strategies - and a scrappy independent blog (open-source) can match them.

Anthropic’s multi-agent research beat a single agent by ~90%. Why is that consistent with 11.5’s multi-agent tax?

**📝 `06_multi_agent.py`** - *illustrative*

In [ ]:
# MULTI-AGENT DEEP RESEARCH - the Anthropic orchestrator-worker (illustrative minimal example).
# A lead agent decomposes and delegates to PARALLEL subagents, each with its own
# clean context; a separate CitationAgent grounds the draft. This beat a single agent
# by ~90% on Anthropic's research eval - at ~15x the tokens (the multi-agent tax, 11.5).
import anthropic, os

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

def subagent(sub_question, tools):    # a worker: searches ONE sub-question in isolation
    return client.messages.create(
        model="claude-sonnet-4-6",    # cheaper workers; the lead uses a stronger model
        max_tokens=2048, tools=tools,
        messages=[{"role": "user", "content": f"Research this and report findings: {sub_question}"}])

def lead_research(question, sub_questions, tools):
    findings = [subagent(sq, tools) for sq in sub_questions]   # run workers in PARALLEL in production
    synthesis = client.messages.create(
        model="claude-opus-4-8", max_tokens=4096,              # the lead synthesises
        messages=[{"role": "user", "content": f"Question: {question}\nWorker findings: {findings}\n"
                                              "Write a cited report; a CitationAgent will verify each citation."}])
    return synthesis

# Reach for multi-agent ONLY when the sub-questions are genuinely independent (research is
# the textbook case). For a single coherent thread (like coding, 11.7), one agent wins.
# Output: (illustrative minimal example - needs anthropic + ANTHROPIC_API_KEY; run workers in parallel.)
# The lead delegates each sub-question to a subagent, then synthesises a cited report;
# a CitationAgent verifies the citations (Step 4). ~90% better than a single agent, ~15x tokens.

- A `lead_research` agent decomposes and delegates each sub-question to a `subagent` with its own clean context.
- Workers run in **parallel** (independent sub-questions), then the lead synthesizes; a CitationAgent grounds the draft (Step 4).
- This is Anthropic’s orchestrator-worker: ~90% better than a single agent, at ~15x tokens (the 11.5 multi-agent tax).
- Reach for multi-agent ONLY when the sub-questions are genuinely independent - research is that case; coding (11.7) is not.

#### 💡 What just happened

⚡ What just happened? Four vendors run the same loop with four bets: reasoning depth (OpenAI), user control (Google), a parallel team (Anthropic), speed (Perplexity); open-source matches them. Tradeoff: multi-agent buys ~90% quality at ~15x tokens - worth it here because research sub-tasks are independent (11.5). Pick the bet that fits your latency and budget.

- Compare single-agent / plan-first / multi-agent / speed
- See cost, time, and quality, and the multi-agent tax

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Cost & Stopping: Making It Shippable

### Cost & Stopping: Making It Shippable

The iterative loop must know when to STOP - and the cost must be controlled, or a query runs forever and bills a fortune.

Two things separate a demo from a production deep-research agent. First, **stopping criteria**: the iterative loop (search, analyze the gaps, search again - Reflexion from 9.2) has no natural end. You must stop it on *coverage* (every sub-question answered), *novelty saturation* (a round found nothing new), and a hard *max-iteration and budget cap*. Second, **cost**: an unoptimized query can cost several dollars; a real report might run **$5-8** naive. The standard stack cuts it by an order of magnitude - **semantic caching** (skip near-duplicate queries), **model routing** (cheap models for decomposition and extraction, a frontier model only for final synthesis), and **prompt caching** (a big discount on repeated context) - bringing it to well under a dollar. The block runs the stopping logic and the cost model together.

> 🏁 **Analogy**
>
> It is **knowing when a research project is done**. A good researcher does not read forever - they stop when every question is answered, when new sources stop adding anything, or when the deadline and budget say so. And they do not fly first-class for every errand: cheap models for the legwork, the expensive one only for the final write-up. Deep research needs both instincts, in code.

An iterative research loop with no stopping criteria and no budget cap. What happens?

**📝 `07_cost_stopping.py`** - *runnable*

In [ ]:
# COST + STOPPING - the iterative loop must know when to STOP, and cost must be
# controlled or a deep-research query runs forever and bills a fortune. Stop on
# coverage / no-new-information / a max-iteration cap; cut cost with caching + routing.

def enough(covered, total, new_findings, iteration, max_iters=4):
    if iteration >= max_iters:            return "STOP: max iterations reached"
    if covered >= total:                  return "STOP: all sub-questions covered"
    if new_findings == 0:                 return "STOP: no new information (saturated)"
    return "CONTINUE"

# Simulate the loop: each round covers more and eventually stops finding new things.
rounds = [(2, 4, 3), (4, 4, 2), (4, 4, 0)]     # (covered, total, new_findings) per round
print("Iterative loop with stopping criteria:")
for i, (cov, tot, new) in enumerate(rounds, 1):
    decision = enough(cov, tot, new, i)
    print(f"  round {i}: covered {cov}/{tot}, new findings {new} -> {decision}")
    if decision.startswith("STOP"):
        break

# Cost model for one query: naive vs the standard stack (cache + route + prompt-cache).
def cost(searches, synth_tokens_k, search_unit, cached, routed, prompt_cache):
    search_cost = searches * search_unit                          # cheap SERP vs premium AI-native
    synth_cost = synth_tokens_k * (0.20 if not routed else 0.04)  # route cheap models for sub-tasks
    if prompt_cache: synth_cost *= 0.5                            # discount on repeated context
    total = search_cost + synth_cost
    if cached: total *= 0.7                                       # semantic cache skips ~30% of queries
    return round(total, 2)

naive     = cost(searches=40, synth_tokens_k=30, search_unit=0.02,  cached=False, routed=False, prompt_cache=False)
optimized = cost(searches=40, synth_tokens_k=30, search_unit=0.002, cached=True,  routed=True,  prompt_cache=True)
print(f"\nCost per query: naive ${naive}  ->  cached+routed+prompt-cache ${optimized}"
      f"  (~{round(naive/optimized)}x cheaper)")
# Output:
# Iterative loop with stopping criteria:
#   round 1: covered 2/4, new findings 3 -> CONTINUE
#   round 2: covered 4/4, new findings 2 -> STOP: all sub-questions covered
#
# Cost per query: naive $6.8  ->  cached+routed+prompt-cache $0.48  (~14x cheaper)

- `enough` stops the loop on coverage, novelty saturation, or a `max_iters` cap - the Reflexion loop with a brake.
- The simulated loop stops at round 2 when all sub-questions are covered, not by running out of money.
- `cost` models the stack: a naive query at ~$6.80 drops to ~$0.48 with caching + model routing + prompt caching.
- That is ~14x cheaper - the real-world “$5-8 to under $0.50” story, made concrete.

#### 💡 What just happened

⚡ What just happened? Shipping deep research is engineering: stopping criteria (coverage / novelty / max-iters / budget) so the loop ends, and a cost stack (semantic caching + model routing + prompt caching) so a query costs cents, not dollars. Tradeoff: caps risk stopping a hair early, but an uncapped loop is a runaway bill. Cap it, cache it, route it.

- Run the iterative loop; stopping criteria fire (coverage / no novelty / max-iters)
- Caching and routing collapse the cost per query

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> A deep-research agent is a **loop** - decompose, search, evaluate, extract, cross-check, synthesize - that returns a cited answer, not links (Step 1). Decomposition is also a **plan**: independent sub-questions run in parallel, dependent ones in order (Step 2). Search picks an API (Bing’s is gone; AI-native vs SERP vs reader) and **scores sources** by credibility and recency (Step 3). The **grounding spine** - store chunks, reference ids, verify each claim - is what keeps the report from being confidently fabricated (Step 4). **Synthesis** is map-reduce with a faithfulness tax, so quote critical facts verbatim (Step 5). The **four vendors** run this loop with different bets, and multi-agent wins here because research is independent work (Step 6). And **cost + stopping** make it shippable: cap the loop, cache and route the calls (Step 7).

| Architecture | Strategy | Time | Best for |
|---|---|---|---|
| OpenAI Deep Research | single agent, RL-trained reasoning depth | 5-30 min | hard, deep questions |
| Google Gemini | user-approved plan + Workspace | 5-15 min | control + Google-hosted sources |
| Anthropic | multi-agent orchestrator-worker + CitationAgent | up to 45 min, ~90% better | broad, parallelizable research |
| Perplexity | speed + pay-as-you-go API | under 3 min | fast answers, an API you can call |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now build a deep-research agent’s loop, ground its citations, synthesize faithfully, and cost it. Module 11 continues in Lesson 11.9. We come back to **evaluating** research agents (RAGAS, faithfulness metrics) in Lesson 11.11 and in Module 14, and to Indian-language search (IndicTrans2, data.gov.in) in Lesson 17.3. The primary references are Anthropic’s [multi-agent research system](https://www.anthropic.com/engineering/multi-agent-research-system), [OpenAI Deep Research](https://openai.com/index/introducing-deep-research/), and [LangChain Open Deep Research](https://github.com/langchain-ai/open_deep_research).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Deep ResearchAgents**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-11_8.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-11.8.html` - regenerate this notebook whenever the source HTML is updated.*
